[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-01-transformer-block-four-facts.ipynb)

# Advanced Discussion 1 — the square-root scaling, order blindness, the quadratic bill and residual connections

Four measurements on attention and the transformer block: why scores are divided by sqrt(d), why positions are needed, how the score matrix grows, and gradient flow with and without residuals.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

CPU only; runs in about a minute.

In [ ]:
%pip install -q torch numpy

In [ ]:
import math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)

## 1. why divide by sqrt(d): dot products of random vectors grow with dimension, and the softmax saturates

In [ ]:
print("q and k with independent N(0,1) entries; 64 keys per query; 2,000 queries")
print("  d      std of q.k   (theory sqrt(d))   mean largest attention weight: raw | scaled     mean gradient size through the softmax: raw | scaled")
def jac_norm(scores):
    p = F.softmax(scores, -1); J = torch.diag_embed(p) - p.unsqueeze(-1) * p.unsqueeze(-2); return J.abs().sum((-1, -2)).mean().item()
for d in (16, 64, 256, 1024):
    q = torch.randn(2000, d); k = torch.randn(2000, 64, d); s = torch.einsum("bd,bkd->bk", q, k)
    raw, sc = s, s / math.sqrt(d)
    print("  %4d   %7.2f      %7.2f             %.3f | %.3f                                     %.4f | %.4f" % (d, s.std(), math.sqrt(d), F.softmax(raw, -1).max(-1).values.mean(), F.softmax(sc, -1).max(-1).values.mean(), jac_norm(raw), jac_norm(sc)))
print("  a softmax that is almost one-hot has almost no gradient: the second number in the last pair stays healthy, the first collapses as d grows")

## 2. self-attention has no idea about order: permute the tokens and the outputs are permuted the same way

In [ ]:
class Attn(nn.Module):
    def __init__(s, d, h):
        super().__init__(); s.h, s.d = h, d; s.qkv = nn.Linear(d, 3 * d); s.o = nn.Linear(d, d)
    def forward(s, x):
        B, T, D = x.shape; q, k, v = s.qkv(x).view(B, T, 3, s.h, D // s.h).unbind(2)
        att = F.softmax(torch.einsum("bthd,bshd->bhts", q, k) / math.sqrt(D // s.h), -1); return s.o(torch.einsum("bhts,bshd->bthd", att, v).reshape(B, T, D))
D, T = 32, 10; attn = Attn(D, 4).eval(); x = torch.randn(1, T, D); perm = torch.randperm(T)
with torch.no_grad(): y = attn(x); yp = attn(x[:, perm])
print("\nwithout positions: max |attn(permuted input) - permuted attn(input)| = %.2e   (the layer cannot tell the order)" % (yp - y[:, perm]).abs().max().item())
def sinusoidal(T, D):
    pos = torch.arange(T).unsqueeze(1).float(); i = torch.arange(0, D, 2).float(); ang = pos / (10000 ** (i / D)); pe = torch.zeros(T, D); pe[:, 0::2] = ang.sin(); pe[:, 1::2] = ang.cos(); return pe
pe = sinusoidal(T, D)
with torch.no_grad():
    a = attn((x + pe)); b = attn((x[:, perm] + pe))
    a2 = attn(x[:, perm] + pe[perm])
print("with sinusoidal positions added: pooled output of 'sentence' vs the same tokens in shuffled order differ by %.3f (was %.1e without positions)" % ((a.mean(1) - b.mean(1)).abs().max().item(), (y.mean(1) - yp.mean(1)).abs().max().item()))
print("(shuffling tokens but carrying each token's own original position along restores equivariance: %.1e)" % (a2 - a[:, perm]).abs().max().item())
# the sinusoidal code has a linear 'move by k positions' map: PE(p + k) = R_k PE(p)
Tbig = 200; PE = sinusoidal(Tbig, D)
for k in (1, 5, 20):
    A, B = PE[:Tbig - k], PE[k:]; W = torch.linalg.lstsq(A, B).solution; res = (A @ W - B).abs().max().item()
    print("  offset k=%2d: one linear map takes PE(p) to PE(p+k) for all %d positions, max error %.1e" % (k, Tbig - k, res))
learned = nn.Embedding(512, D); print("  a learned table with 512 rows has no row 600: extrapolation is undefined; the sinusoidal formula can produce any position (whether the model uses it well beyond training lengths is another matter)")

## 3. the price of attending to everything: the n x n score matrix

In [ ]:
print("\nattention score matrix, one head, one sequence, float32")
print("  n      matrix size    time to compute softmax(QK^T)V (ms)    time ratio to previous")
prev = None
for n in (256, 512, 1024, 2048, 4096):
    q = torch.randn(n, 64); k = torch.randn(n, 64); v = torch.randn(n, 64)
    for _ in range(2): F.softmax(q @ k.T / 8, -1) @ v
    t0 = time.perf_counter()
    for _ in range(5): F.softmax(q @ k.T / 8, -1) @ v
    t = (time.perf_counter() - t0) / 5 * 1000
    print("  %4d   %7.1f MB     %8.2f                              %s" % (n, 4 * n * n / 1e6, t, "-" if prev is None else "%.1fx" % (t / prev))); prev = t
print("  doubling n multiplies the matrix by exactly 4; with 32 heads x 32 layers a 4,096-token prompt would materialise %.1f GB of scores if nothing were fused" % (32 * 32 * 4 * 4096 * 4096 / 1e9))

## 4. why residual connections: gradient reaching the input of a deep stack

In [ ]:
def grad_at_input(depth, residual, norm):
    torch.manual_seed(1); layers = [nn.Sequential(nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 64)) for _ in range(depth)]; norms = [nn.LayerNorm(64) for _ in range(depth)]
    x = torch.randn(32, 64, requires_grad=True); h = x
    for L, N in zip(layers, norms):
        f = L(N(h) if norm else h); h = h + f if residual else f
    h.pow(2).mean().backward(); return x.grad.norm().item()
print("\ngradient norm at the input of a stack of depth L (same random init), loss = mean(output^2)")
print("  depth   plain stack     with residual connections    residual + pre-norm")
for L in (2, 8, 24, 48):
    print("  %4d    %.2e        %.2e                  %.2e" % (L, grad_at_input(L, False, False), grad_at_input(L, True, False), grad_at_input(L, True, True)))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.